# Porto Taxi Long-Trip ETA Notebook

Notebook nay tach rieng bai toan `long trip ETA` de kiem tra xem model chuyen biet co giup du doan cac chuyen dai tot hon hay khong.


In [1]:
import ast
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

DATA_PATH = Path("../train.csv")
SAMPLE_ROWS = 100000

LONG_TRIP_THRESHOLD_SECONDS = 3600
MIN_PREFIX_POINTS = 3
PREFIX_STEP = 5
MAX_TRIPS_FOR_PREFIX = 100000

TOP_6_FEATURES = [
    "observed_points",
    "distance_to_destination",
    "distance_from_start",
    "progress_ratio",
    "recent_speed_avg_kmh",
    "avg_speed_since_start_kmh",
]

df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)
print(f"Loaded {len(df):,} rows from {DATA_PATH.resolve()}")
df.head(3)


Loaded 100,000 rows from /home/duy/ml_project/train.csv


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,NaN,NaN,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,NaN,7.0,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,NaN,NaN,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."


## Parse `POLYLINE` va lam sach du lieu

Buoc nay giong notebook chinh: parse trajectory, bo `MISSING_DATA=True`, va tao cac feature thoi gian co ban.


In [2]:
def parse_polyline(polyline_text: str):
    points = ast.literal_eval(polyline_text)
    return points if isinstance(points, list) else []


sample = df.loc[:, [
    "TRIP_ID",
    "TAXI_ID",
    "TIMESTAMP",
    "MISSING_DATA",
    "POLYLINE",
]].copy()
sample["parsed_polyline"] = sample["POLYLINE"].apply(parse_polyline)
sample["num_points"] = sample["parsed_polyline"].apply(len)
sample["trip_duration_seconds"] = (sample["num_points"] - 1).clip(lower=0) * 15

clean_df = sample.copy()
clean_df["missing_flag"] = clean_df["MISSING_DATA"].astype(str).str.lower() == "true"
clean_df = clean_df[(~clean_df["missing_flag"]) & (clean_df["num_points"] > 1)].copy()

clean_df["start_datetime"] = pd.to_datetime(clean_df["TIMESTAMP"], unit="s")
clean_df["start_hour"] = clean_df["start_datetime"].dt.hour
clean_df["day_of_week"] = clean_df["start_datetime"].dt.dayofweek
clean_df["is_weekend"] = clean_df["day_of_week"].isin([5, 6])

print(f"Clean rows kept: {len(clean_df):,} / {len(sample):,}")
display(clean_df[[
    "TRIP_ID",
    "start_datetime",
    "num_points",
    "trip_duration_seconds",
    "start_hour",
    "day_of_week",
    "is_weekend",
]].head(10))


Clean rows kept: 98,445 / 100,000


,TRIP_ID,start_datetime,num_points,trip_duration_seconds,start_hour,day_of_week,is_weekend
0,1372636858620000589,2013-07-01 00:00:58,23,330,0,0,False
1,1372637303620000596,2013-07-01 00:08:23,19,270,0,0,False
2,1372636951620000320,2013-07-01 00:02:31,65,960,0,0,False
3,1372636854620000520,2013-07-01 00:00:54,43,630,0,0,False
4,1372637091620000337,2013-07-01 00:04:51,29,420,0,0,False
5,1372636965620000231,2013-07-01 00:02:45,26,375,0,0,False
6,1372637210620000456,2013-07-01 00:06:50,36,525,0,0,False
7,1372637299620000011,2013-07-01 00:08:19,34,495,0,0,False
8,1372637274620000403,2013-07-01 00:07:54,38,555,0,0,False
9,1372637905620000320,2013-07-01 00:18:25,19,270,0,0,False


## Tao feature trajectory cho bai toan ETA

Cell nay dung lai bo feature da on dinh hon o notebook chinh, gom khoang cach, tien do hanh trinh, va toc do gan day.


In [3]:
def haversine_distance_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    earth_radius_km = 6371.0
    return earth_radius_km * c


def build_prefix_samples(points, trip_id, min_prefix_points=3, step=5):
    rows = []
    total_points = len(points)
    start_lon, start_lat = points[0]
    dest_lon, dest_lat = points[-1]

    for prefix_points in range(min_prefix_points, total_points, step):
        observed = points[:prefix_points]
        remaining_segments = total_points - prefix_points
        elapsed_seconds = (prefix_points - 1) * 15
        current_lon, current_lat = observed[-1]

        distance_to_destination = haversine_distance_km(current_lon, current_lat, dest_lon, dest_lat)
        distance_from_start = haversine_distance_km(start_lon, start_lat, current_lon, current_lat)
        total_trip_distance_proxy = distance_from_start + distance_to_destination
        progress_ratio = distance_from_start / total_trip_distance_proxy if total_trip_distance_proxy > 0 else 0.0
        avg_speed_since_start_kmh = distance_from_start / (elapsed_seconds / 3600)

        prev_lon, prev_lat = observed[-2]
        recent_distance_km = haversine_distance_km(prev_lon, prev_lat, current_lon, current_lat)
        recent_speed_kmh = recent_distance_km / (15 / 3600)

        segment_speeds = []
        recent_points = observed[-4:]
        for i in range(1, len(recent_points)):
            lon_a, lat_a = recent_points[i - 1]
            lon_b, lat_b = recent_points[i]
            segment_distance_km = haversine_distance_km(lon_a, lat_a, lon_b, lat_b)
            segment_speed_kmh = segment_distance_km / (15 / 3600)
            segment_speeds.append(segment_speed_kmh)

        recent_speed_avg_kmh = sum(segment_speeds) / len(segment_speeds)

        rows.append({
            "TRIP_ID": trip_id,
            "observed_points": prefix_points,
            "remaining_points": remaining_segments,
            "remaining_eta_seconds": remaining_segments * 15,
            "distance_to_destination": distance_to_destination,
            "distance_from_start": distance_from_start,
            "progress_ratio": progress_ratio,
            "avg_speed_since_start_kmh": avg_speed_since_start_kmh,
            "recent_speed_kmh": recent_speed_kmh,
            "recent_speed_avg_kmh": recent_speed_avg_kmh,
        })

    return rows


example_trip = clean_df.iloc[0]
example_prefix_df = pd.DataFrame(
    build_prefix_samples(example_trip["parsed_polyline"], example_trip["TRIP_ID"])
)
print(f"Example trip id: {example_trip['TRIP_ID']}")
display(example_prefix_df.head(10))


Example trip id: 1372636858620000589


,TRIP_ID,observed_points,remaining_points,remaining_eta_seconds,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh
0,1372636858620000589,3,20,300,1.596541,0.186463,0.104578,22.375556,47.581896,25.315617
1,1372636858620000589,8,15,225,1.033049,1.057881,0.505938,36.270200,58.946877,42.966445
2,1372636858620000589,13,10,150,0.388607,1.401802,0.782951,28.036039,0.806232,32.172979
3,1372636858620000589,18,5,75,0.205102,1.725886,0.893784,24.365443,31.204171,25.769688


## Tao prefix dataset tu nhieu trip

Buoc nay bien moi trip thanh nhieu sample `prefix -> remaining_eta_seconds`, giong bai toan ETA theo thoi gian thuc.


In [4]:
prefix_rows = []
selected_trips = clean_df.head(MAX_TRIPS_FOR_PREFIX)

for trip in selected_trips.itertuples(index=False):
    trip_prefixes = build_prefix_samples(
        points=trip.parsed_polyline,
        trip_id=trip.TRIP_ID,
        min_prefix_points=MIN_PREFIX_POINTS,
        step=PREFIX_STEP,
    )

    for row in trip_prefixes:
        row["TAXI_ID"] = trip.TAXI_ID
        row["start_hour"] = trip.start_hour
        row["day_of_week"] = trip.day_of_week
        row["is_weekend"] = trip.is_weekend
        row["trip_duration_seconds"] = trip.trip_duration_seconds
        prefix_rows.append(row)

prefix_dataset = pd.DataFrame(prefix_rows)

print(f"Trips used: {len(selected_trips):,}")
print(f"Prefix samples created: {len(prefix_dataset):,}")
display(prefix_dataset.head(10))


Trips used: 98,445
Prefix samples created: 934,906


,TRIP_ID,observed_points,remaining_points,remaining_eta_seconds,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,TAXI_ID,start_hour,day_of_week,is_weekend,trip_duration_seconds
0,1372636858620000589,3,20,300,1.596541,0.186463,0.104578,22.375556,47.581896,25.315617,20000589,0,0,False,330
1,1372636858620000589,8,15,225,1.033049,1.057881,0.505938,36.270200,58.946877,42.966445,20000589,0,0,False,330
2,1372636858620000589,13,10,150,0.388607,1.401802,0.782951,28.036039,0.806232,32.172979,20000589,0,0,False,330
3,1372636858620000589,18,5,75,0.205102,1.725886,0.893784,24.365443,31.204171,25.769688,20000589,0,0,False,330
4,1372637303620000596,3,16,240,2.293955,0.199239,0.079913,23.908689,37.632331,23.914791,20000596,0,0,False,270
5,1372637303620000596,8,11,165,1.190055,1.421936,0.544388,48.752083,67.252865,65.337089,20000596,0,0,False,270
6,1372637303620000596,13,6,90,0.749598,2.661823,0.780268,53.236467,35.723373,53.467636,20000596,0,0,False,270
7,1372637303620000596,18,1,15,0.004597,2.480394,0.998150,35.017328,24.294560,35.039908,20000596,0,0,False,270
8,1372636951620000320,3,62,930,0.149613,0.105146,0.412729,12.617570,16.931268,12.627616,20000320,0,0,False,960
9,1372636951620000320,8,57,855,0.463589,0.695954,0.600197,23.861291,56.737664,38.219668,20000320,0,0,False,960


## Chot dataset train va bo `top_6_features`

Notebook nay chi tap trung vao bo feature da cho ket qua tot hon o notebook chinh, de tranh phan tan qua nhieu nhanh thu nghiem.


In [5]:
baseline_columns = [
    "TRIP_ID",
    "TAXI_ID",
    "observed_points",
    "remaining_points",
    "distance_to_destination",
    "distance_from_start",
    "progress_ratio",
    "avg_speed_since_start_kmh",
    "recent_speed_kmh",
    "recent_speed_avg_kmh",
    "start_hour",
    "day_of_week",
    "is_weekend",
    "trip_duration_seconds",
    "remaining_eta_seconds",
]

baseline_df = prefix_dataset[baseline_columns].copy()
print("Baseline dataset shape:", baseline_df.shape)
print("Top-6 features:", TOP_6_FEATURES)
display(baseline_df.head(10))


Baseline dataset shape: (934906, 15)
Top-6 features: ['observed_points', 'distance_to_destination', 'distance_from_start', 'progress_ratio', 'recent_speed_avg_kmh', 'avg_speed_since_start_kmh']


,TRIP_ID,TAXI_ID,observed_points,remaining_points,distance_to_destination,distance_from_start,progress_ratio,avg_speed_since_start_kmh,recent_speed_kmh,recent_speed_avg_kmh,start_hour,day_of_week,is_weekend,trip_duration_seconds,remaining_eta_seconds
0,1372636858620000589,20000589,3,20,1.596541,0.186463,0.104578,22.375556,47.581896,25.315617,0,0,False,330,300
1,1372636858620000589,20000589,8,15,1.033049,1.057881,0.505938,36.270200,58.946877,42.966445,0,0,False,330,225
2,1372636858620000589,20000589,13,10,0.388607,1.401802,0.782951,28.036039,0.806232,32.172979,0,0,False,330,150
3,1372636858620000589,20000589,18,5,0.205102,1.725886,0.893784,24.365443,31.204171,25.769688,0,0,False,330,75
4,1372637303620000596,20000596,3,16,2.293955,0.199239,0.079913,23.908689,37.632331,23.914791,0,0,False,270,240
5,1372637303620000596,20000596,8,11,1.190055,1.421936,0.544388,48.752083,67.252865,65.337089,0,0,False,270,165
6,1372637303620000596,20000596,13,6,0.749598,2.661823,0.780268,53.236467,35.723373,53.467636,0,0,False,270,90
7,1372637303620000596,20000596,18,1,0.004597,2.480394,0.998150,35.017328,24.294560,35.039908,0,0,False,270,15
8,1372636951620000320,20000320,3,62,0.149613,0.105146,0.412729,12.617570,16.931268,12.627616,0,0,False,960,930
9,1372636951620000320,20000320,8,57,0.463589,0.695954,0.600197,23.861291,56.737664,38.219668,0,0,False,960,855


## Tach rieng tap `long trip`

O day, `long trip` duoc dinh nghia la sample co `remaining_eta_seconds >= 3600` giay, tuc ETA con lai tu `60 phut` tro len.


In [6]:
long_trip_df = baseline_df[baseline_df["remaining_eta_seconds"] >= LONG_TRIP_THRESHOLD_SECONDS].copy()

print(f"Long-trip samples: {len(long_trip_df):,}")
print(f"Unique long-trip trips: {long_trip_df['TRIP_ID'].nunique():,}")
display(long_trip_df["remaining_eta_seconds"].describe())


Long-trip samples: 19,413
Unique long-trip trips: 550


count    19413.000000
mean      7981.649668
std       5621.587859
min       3600.000000
25%       4395.000000
50%       5760.000000
75%       9180.000000
max      37695.000000
Name: remaining_eta_seconds, dtype: float64

## Tao train-validation split cho long trips

Validation se chi gom long trips. Sau do ta dung cung mot tap validation nay de so sanh:
- `global model`: train tren du lieu tong quat
- `specialized model`: train chi tren long trips


In [7]:
long_groups = long_trip_df["TRIP_ID"].copy()
long_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
long_train_idx, long_valid_idx = next(
    long_splitter.split(long_trip_df[TOP_6_FEATURES], long_trip_df["remaining_eta_seconds"], groups=long_groups)
)

long_train_df = long_trip_df.iloc[long_train_idx].copy()
long_valid_df = long_trip_df.iloc[long_valid_idx].copy()
long_valid_trip_ids = set(long_valid_df["TRIP_ID"].unique())

global_train_df = baseline_df[~baseline_df["TRIP_ID"].isin(long_valid_trip_ids)].copy()

print("Global-train samples:", len(global_train_df))
print("Long-train samples:", len(long_train_df))
print("Long-valid samples:", len(long_valid_df))
print("Long-valid unique trips:", long_valid_df["TRIP_ID"].nunique())


Global-train samples: 926088
Long-train samples: 15852
Long-valid samples: 3561
Long-valid unique trips: 110


## Train `global XGBoost log-target` va test tren long-trip validation

Cell nay giu vai tro moc so sanh. Model duoc train tren toan bo train data, nhung chi danh gia tren long trips.


In [8]:
X_global_train = global_train_df[TOP_6_FEATURES]
y_global_train = global_train_df["remaining_eta_seconds"]
X_long_valid = long_valid_df[TOP_6_FEATURES]
y_long_valid = long_valid_df["remaining_eta_seconds"]

global_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

global_model.fit(X_global_train, np.log1p(y_global_train))
global_long_predictions = np.expm1(global_model.predict(X_long_valid))
global_long_predictions = np.clip(global_long_predictions, a_min=0, a_max=None)

global_long_mae = mean_absolute_error(y_long_valid, global_long_predictions)
global_long_rmse = mean_squared_error(y_long_valid, global_long_predictions) ** 0.5

print(f"Global model on long-trip valid MAE: {global_long_mae:.2f} seconds ({global_long_mae / 60:.2f} minutes)")
print(f"Global model on long-trip valid RMSE: {global_long_rmse:.2f} seconds ({global_long_rmse / 60:.2f} minutes)")


Global model on long-trip valid MAE: 5124.71 seconds (85.41 minutes)
Global model on long-trip valid RMSE: 6464.23 seconds (107.74 minutes)


## Train `specialized long-trip XGBoost log-target`

Cell nay chi train tren long trips, roi cung danh gia tren chinh tap long-trip validation o tren.


In [9]:
X_long_train = long_train_df[TOP_6_FEATURES]
y_long_train = long_train_df["remaining_eta_seconds"]

specialized_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

specialized_model.fit(X_long_train, np.log1p(y_long_train))
specialized_predictions = np.expm1(specialized_model.predict(X_long_valid))
specialized_predictions = np.clip(specialized_predictions, a_min=0, a_max=None)

specialized_mae = mean_absolute_error(y_long_valid, specialized_predictions)
specialized_rmse = mean_squared_error(y_long_valid, specialized_predictions) ** 0.5

comparison_summary_df = pd.DataFrame([
    {
        "model": "global_log_top6",
        "mae_seconds": global_long_mae,
        "rmse_seconds": global_long_rmse,
        "mae_minutes": global_long_mae / 60,
        "rmse_minutes": global_long_rmse / 60,
    },
    {
        "model": "specialized_long_log_top6",
        "mae_seconds": specialized_mae,
        "rmse_seconds": specialized_rmse,
        "mae_minutes": specialized_mae / 60,
        "rmse_minutes": specialized_rmse / 60,
    },
])

print(f"Specialized model on long-trip valid MAE: {specialized_mae:.2f} seconds ({specialized_mae / 60:.2f} minutes)")
print(f"Specialized model on long-trip valid RMSE: {specialized_rmse:.2f} seconds ({specialized_rmse / 60:.2f} minutes)")
display(comparison_summary_df)


Specialized model on long-trip valid MAE: 2841.04 seconds (47.35 minutes)
Specialized model on long-trip valid RMSE: 4312.87 seconds (71.88 minutes)


,model,mae_seconds,rmse_seconds,mae_minutes,rmse_minutes
0,global_log_top6,5124.709961,6464.225862,85.411833,107.737098
1,specialized_long_log_top6,2841.042236,4312.871201,47.350704,71.881187


## So sanh tung dong du doan tren long trips

Cell nay gom du doan thuc te va sai so cua hai model tren cung tap validation long trips.


In [10]:
long_trip_comparison_df = X_long_valid.copy()
long_trip_comparison_df["actual_eta_seconds"] = y_long_valid.values
long_trip_comparison_df["global_prediction_seconds"] = global_long_predictions
long_trip_comparison_df["specialized_prediction_seconds"] = specialized_predictions
long_trip_comparison_df["global_absolute_error_seconds"] = (
    long_trip_comparison_df["actual_eta_seconds"] - long_trip_comparison_df["global_prediction_seconds"]
).abs()
long_trip_comparison_df["specialized_absolute_error_seconds"] = (
    long_trip_comparison_df["actual_eta_seconds"] - long_trip_comparison_df["specialized_prediction_seconds"]
).abs()
long_trip_comparison_df["error_delta_seconds"] = (
    long_trip_comparison_df["specialized_absolute_error_seconds"] - long_trip_comparison_df["global_absolute_error_seconds"]
)

display(long_trip_comparison_df.head(10))


,observed_points,distance_to_destination,distance_from_start,progress_ratio,recent_speed_avg_kmh,avg_speed_since_start_kmh,actual_eta_seconds,global_prediction_seconds,specialized_prediction_seconds,global_absolute_error_seconds,specialized_absolute_error_seconds,error_delta_seconds
11014,3,0.206223,0.208941,0.503274,25.073753,25.072927,5430,629.780579,4490.888184,4800.219421,939.111816,-3861.107605
11015,8,0.420198,0.428968,0.505164,23.327216,14.707471,5355,350.608643,5210.083984,5004.391357,144.916016,-4859.475342
11016,13,0.521825,0.534099,0.505812,11.411910,10.681985,5280,356.181641,5165.287109,4923.818359,114.712891,-4809.105469
11017,18,0.610023,0.622758,0.505165,9.887757,8.791881,5205,376.782837,5514.288086,4828.217163,309.288086,-4518.929077
11018,23,0.608593,0.621320,0.505174,0.240564,6.778033,5130,608.654480,4801.452148,4521.345520,328.547852,-4192.797668
11019,28,0.610139,0.622862,0.505159,0.385867,5.536552,5055,656.275269,4963.145508,4398.724731,91.854492,-4306.870239
11020,33,0.612440,0.625159,0.505139,0.241138,4.688692,4980,715.907532,4814.578125,4264.092468,165.421875,-4098.670593
11021,38,0.610934,0.623653,0.505151,0.321198,4.045316,4905,720.757263,4768.566895,4184.242737,136.433105,-4047.809631
11022,43,0.608012,0.620722,0.505172,0.469094,3.546981,4830,680.565674,4847.947266,4149.434326,17.947266,-4131.487061
11023,48,0.623260,0.635956,0.505041,0.060285,3.247437,4755,681.518921,4687.904785,4073.481079,67.095215,-4006.385864


## Xem cac long-trip case te nhat cua model chuyen biet

Buoc nay giup ban xem model specialized van dang gap kho o nhung case nao.


In [11]:
worst_specialized_rows = long_trip_comparison_df.sort_values(
    "specialized_absolute_error_seconds", ascending=False
).head(20)

display(
    worst_specialized_rows[[
        "actual_eta_seconds",
        "global_prediction_seconds",
        "specialized_prediction_seconds",
        "global_absolute_error_seconds",
        "specialized_absolute_error_seconds",
        "error_delta_seconds",
    ]]
)


,actual_eta_seconds,global_prediction_seconds,specialized_prediction_seconds,global_absolute_error_seconds,specialized_absolute_error_seconds,error_delta_seconds
366955,4785,4640.553223,25122.917969,144.446777,20337.917969,20193.471191
410184,3720,549.021851,24034.105469,3170.978149,20314.105469,17143.127319
410185,3645,520.206299,22855.855469,3124.793701,19210.855469,16086.061768
410181,3945,1500.298218,22478.800781,2444.701782,18533.800781,16089.098999
410177,4245,1840.044434,22680.230469,2404.955566,18435.230469,16030.274902
410178,4170,1359.697388,21999.013672,2810.302612,17829.013672,15018.711060
241331,22830,330.735962,6010.022949,22499.264038,16819.977051,-5679.286987
241334,22605,486.361572,6064.186035,22118.638428,16540.813965,-5577.824463
241332,22755,409.504517,6217.327148,22345.495483,16537.672852,-5807.822632
410175,4395,1414.794189,19573.123047,2980.205811,15178.123047,12197.917236


## So sanh metric theo 2 nhom long trips

Cell nay chia long-trip validation thanh 2 regime:
- `60-120 phut`
- `>120 phut`

Muc tieu la xem model specialized dang giup o nhom nao nhieu hon.


In [12]:
bin_labels = ["60_120m", "120m_plus"]
long_trip_comparison_df["eta_regime"] = pd.cut(
    long_trip_comparison_df["actual_eta_seconds"],
    bins=[3600, 7200, float("inf")],
    labels=bin_labels,
    right=False,
)

regime_rows = []
for regime, regime_df in long_trip_comparison_df.groupby("eta_regime", observed=False):
    if len(regime_df) == 0:
        continue

    global_mae = mean_absolute_error(
        regime_df["actual_eta_seconds"], regime_df["global_prediction_seconds"]
    )
    global_rmse = mean_squared_error(
        regime_df["actual_eta_seconds"], regime_df["global_prediction_seconds"]
    ) ** 0.5

    specialized_mae = mean_absolute_error(
        regime_df["actual_eta_seconds"], regime_df["specialized_prediction_seconds"]
    )
    specialized_rmse = mean_squared_error(
        regime_df["actual_eta_seconds"], regime_df["specialized_prediction_seconds"]
    ) ** 0.5

    regime_rows.append({
        "eta_regime": regime,
        "sample_count": len(regime_df),
        "global_mae_seconds": global_mae,
        "specialized_mae_seconds": specialized_mae,
        "mae_delta_seconds": specialized_mae - global_mae,
        "global_rmse_seconds": global_rmse,
        "specialized_rmse_seconds": specialized_rmse,
        "rmse_delta_seconds": specialized_rmse - global_rmse,
    })

regime_comparison_df = pd.DataFrame(regime_rows)
display(regime_comparison_df)


,eta_regime,sample_count,global_mae_seconds,specialized_mae_seconds,mae_delta_seconds,global_rmse_seconds,specialized_rmse_seconds,rmse_delta_seconds
0,60_120m,2422,3257.126465,1614.176636,-1642.949829,3562.803391,2610.66074,-952.142650
1,120m_plus,1139,9095.988281,5449.881348,-3646.106934,10180.845152,6607.68159,-3573.163561


## Train model rieng cho `very long trips`

Cell nay chi dung nhung sample co `remaining_eta_seconds >= 7200` de train mot model chuyen biet hon cho nhom rat dai.


In [13]:
VERY_LONG_THRESHOLD_SECONDS = 7200

very_long_train_df = long_train_df[long_train_df["remaining_eta_seconds"] >= VERY_LONG_THRESHOLD_SECONDS].copy()
very_long_valid_df = long_valid_df[long_valid_df["remaining_eta_seconds"] >= VERY_LONG_THRESHOLD_SECONDS].copy()

print("Very-long train samples:", len(very_long_train_df))
print("Very-long valid samples:", len(very_long_valid_df))
print("Very-long valid unique trips:", very_long_valid_df["TRIP_ID"].nunique())

X_very_long_train = very_long_train_df[TOP_6_FEATURES]
y_very_long_train = very_long_train_df["remaining_eta_seconds"]
X_very_long_valid = very_long_valid_df[TOP_6_FEATURES]
y_very_long_valid = very_long_valid_df["remaining_eta_seconds"]

very_long_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

very_long_model.fit(X_very_long_train, np.log1p(y_very_long_train))
very_long_predictions = np.expm1(very_long_model.predict(X_very_long_valid))
very_long_predictions = np.clip(very_long_predictions, a_min=0, a_max=None)

specialized_on_very_long = specialized_model.predict(X_very_long_valid)
specialized_on_very_long = np.expm1(specialized_on_very_long)
specialized_on_very_long = np.clip(specialized_on_very_long, a_min=0, a_max=None)

global_on_very_long = global_model.predict(X_very_long_valid)
global_on_very_long = np.expm1(global_on_very_long)
global_on_very_long = np.clip(global_on_very_long, a_min=0, a_max=None)

very_long_summary_df = pd.DataFrame([
    {
        "model": "global_log_top6",
        "mae_seconds": mean_absolute_error(y_very_long_valid, global_on_very_long),
        "rmse_seconds": mean_squared_error(y_very_long_valid, global_on_very_long) ** 0.5,
    },
    {
        "model": "specialized_long_log_top6",
        "mae_seconds": mean_absolute_error(y_very_long_valid, specialized_on_very_long),
        "rmse_seconds": mean_squared_error(y_very_long_valid, specialized_on_very_long) ** 0.5,
    },
    {
        "model": "specialized_very_long_log_top6",
        "mae_seconds": mean_absolute_error(y_very_long_valid, very_long_predictions),
        "rmse_seconds": mean_squared_error(y_very_long_valid, very_long_predictions) ** 0.5,
    },
])

very_long_summary_df["mae_minutes"] = very_long_summary_df["mae_seconds"] / 60
very_long_summary_df["rmse_minutes"] = very_long_summary_df["rmse_seconds"] / 60

display(very_long_summary_df)


Very-long train samples: 5794
Very-long valid samples: 1139
Very-long valid unique trips: 18


,model,mae_seconds,rmse_seconds,mae_minutes,rmse_minutes
0,global_log_top6,9095.988281,10180.845152,151.599805,169.680753
1,specialized_long_log_top6,5449.881348,6607.681590,90.831356,110.128027
2,specialized_very_long_log_top6,3423.840332,4535.190183,57.064006,75.586503
